# Run ERM (SGNM equivariant model) on a GPU

Runtime → **Change runtime type → GPU (T4 is fine)**, then **Runtime → Run all**.

Produces `results/erm_profiles.csv` and downloads it. Back on your Mac:
```
# drop the file into results/, then:
python3 scripts/ingest_erm.py
```
which merges ERM into `scores.csv` and regenerates all plots + `results.html`.

In [ ]:
# 1. System + Python deps
!apt-get -qq install -y gperf >/dev/null
!pip -q install gemmi
!pip -q install git+https://github.com/hmblair/dlu.git
!pip -q install git+https://github.com/hmblair/ciffy.git
# flash-eq: prebuilt CUDA wheels
!pip -q install flash-eq --extra-index-url https://hmblair.github.io/flash-eq/
!pip -q install 'git+https://github.com/hmblair/sgnm.git'
import torch; print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 2. Get this project's data + scripts, and the ERM checkpoint
!git clone -q https://github.com/rhiju/rna-shape-decoys.git
%cd rna-shape-decoys
!curl -sL -o equivariant-checkpoint.pth \
  https://github.com/hmblair/sgnm/releases/download/v2.0.2/equivariant-checkpoint.pth
!ls -lh equivariant-checkpoint.pth

In [ ]:
# 3. Sanity check: ciffy must type atoms correctly (column-aligned converter)
import sys; sys.path.insert(0, 'scripts')
import ciffy, numpy as np, tempfile
from pdb_to_cif import pdb_to_cif
with tempfile.NamedTemporaryFile(suffix='.cif') as tf:
    pdb_to_cif('data/farfar2/Mol9_reference_UtoG_buildloop.pdb', tf.name)
    p = ciffy.load(tf.name, backend='numpy')
a = np.asarray(p.atoms)
print('code-0 atoms (must be 0):', int((a == 0).sum()), '/', len(a))
assert (a == 0).sum() == 0, 'atom typing failed — check pdb_to_cif'

In [ ]:
# 4. Run ERM on all structures
!python3 scripts/run_erm.py equivariant-checkpoint.pth

In [ ]:
# 5. Download results/erm_profiles.csv
from google.colab import files
files.download('results/erm_profiles.csv')